In [1]:
import h5py
from itertools import product
import os
import numpy as np
import nibabel as nib
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import mixedlm

# 加载神经反应数据

# 加载行为数据

file_path1 = 'face10156_z_correlations_real_si.h5'

def load_data(file_path):
    """Function to load data from subarray_0 to subarray_399 from a given file path."""
    with h5py.File(file_path, 'r') as file:
        data_list = []
        for i in range(400):  # Load from subarray_0 to subarray_399
            dataset_name = f'z_corr_matrix_{i}'
            if dataset_name in file:
                data = file[dataset_name][:]
                data_list.append(data)
            else:
                print(f"Dataset '{dataset_name}' not found in the file.")
        return data_list

# Load data for each file
all_z_correlations_real = load_data(file_path1)

msub_r_list = []
for brain_area in range(400):
    z_corr_matrix_real = all_z_correlations_real[brain_area]
    corr_sametrial_stacked = []

    for matrix in z_corr_matrix_real:
        # Extract the diagonal of the submatrices
        submatrix = np.diag(matrix)
        # Append to the list for current brain area
        corr_sametrial_stacked.append(submatrix)

    # Append the list of stacked arrays to the main list
    msub_r_list.append(corr_sametrial_stacked)
    
realpair = np.array(msub_r_list)  # 假设数据保存为.npy文件


In [2]:
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import multiprocessing as mp
from functools import partial
import os
from statsmodels.stats.multitest import fdrcorrection

# FDR校正函数
def fdr_correction(p_values):
    """应用FDR校正到p值列表"""
    _, p_corrected = fdrcorrection(p_values)
    return p_corrected

# 启用 Pandas 与 R DataFrame 互操作
pandas2ri.activate()

# 定义处理单个脑区的函数
def process_region(region, realpair):
    import pandas as pd
    import numpy as np
    import os
    from rpy2.robjects import pandas2ri
    import rpy2.robjects as ro
    pandas2ri.activate()

    process_id = os.getpid()
    print(f"Process {process_id} - Processing region {region}")
    
    try:
        # === 1. 构造数据，每行表示一个 trial ===
        data_list = []
        for subject in range(46):
            for t in range(24):
                condition = 1 if t < 12 else 2
                stimulus_group = 2 if subject >= 23 else 1
                y_real = realpair[region, subject, t]
                data_list.append([subject, t, condition, y_real, stimulus_group])
        
        df = pd.DataFrame(data_list, columns=["Subject", "Trial", "Condition", "y", "StimulusGroup"])
        
        # === 2. Z-score标准化 ===
        df["z_real"] = df.groupby("Subject")["y"].transform(lambda x: (x - x.mean()) / x.std(ddof=0))
        
        # === 3. 剔除离群值 ===
        df["is_outlier"] = df["z_real"].abs() > 3
        df_clean = df[~df["is_outlier"]].copy()
        
        if df_clean.empty:
            print(f"Process {process_id} - Skipping region {region}: all data removed after cleaning")
            return None
        
        # === 4. 准备 R 所需格式 ===
        df_clean["Subject"] = df_clean["Subject"].astype(str)

        # === 5. 转换为 R dataframe ===
        r_df = pandas2ri.py2rpy(df_clean)
        ro.globalenv["df"] = r_df

        # === 6. R代码：只使用置换检验测试y是否显著大于0 ===
        r_code = """
        # 保留 trial-clean 后原始 subject-wise trial mean 向量
        subject_means <- aggregate(y ~ Subject, data=df, FUN=mean)
        subject_original_means <- subject_means$y
        
        # 原始 group mean
        original_mean <- mean(subject_original_means)
        
        N_PERMUTATIONS <- 3000
        perm_means <- numeric(N_PERMUTATIONS)
        
        for (i in 1:N_PERMUTATIONS) {
          permuted_subject_means <- c()
          
          for (subj in unique(df$Subject)) {
            subj_trials <- df[df$Subject == subj, "y"]
            
            # 在该被试的 trial 内部打乱正负号
            flipped_trials <- subj_trials * sample(c(-1, 1), size=length(subj_trials), replace=TRUE)
            
            # 得到该被试的 trial-flipped 平均值
            subj_mean <- mean(flipped_trials)
            
            permuted_subject_means <- c(permuted_subject_means, subj_mean)
          }
          
          # 该次 permutation 的 group-level 平均值
          perm_means[i] <- mean(permuted_subject_means)
        }
        
        # 计算置换 p 值
        p_value_perm <- mean(perm_means >= original_mean)
        
        # 计算效应量: Cohen's d（基于原始被试均值）
        cohens_d <- original_mean / sd(subject_original_means)
        
        list(
            mean_y = original_mean,
            p_value_perm = p_value_perm,
            cohens_d = cohens_d
        )
        """

        
        r_results = ro.r(r_code)
        results = [region] + list(r_results)
        print(f"Process {process_id} - Completed region {region}")
        return results

    except Exception as e:
        print(f"Process {process_id} - Error processing region {region}: {e}")
        return None


# 主函数 - 并行处理
def run_analysis(realpair):
    # 设置并行处理的核心数
    num_cores = 8
    print(f"Running with {num_cores} cores")
    
    # 创建进程池
    pool = mp.Pool(processes=num_cores)
    
    # 创建偏函数，只传入realpair参数
    process_func = partial(process_region, realpair=realpair)
    
    # 并行处理所有脑区
    results = pool.map(process_func, range(400))
    
    # 关闭进程池
    pool.close()
    pool.join()
    
    # 过滤掉None值
    results_list = [r for r in results if r is not None]
    
    # 创建结果DataFrame
    df_results = pd.DataFrame(results_list, columns=[
        "Brain_Region", "Mean_Y", "P_Value_Perm", "Cohens_D"
    ])
    
    return df_results

# 使用方法:
results = run_analysis(np.array(realpair))

Running with 8 cores
Process 586247 - Processing region 0
Process 586248 - Processing region 13
Process 586249 - Processing region 26
Process 586250 - Processing region 39
Process 586251 - Processing region 52
Process 586252 - Processing region 65
Process 586253 - Processing region 78
Process 586254 - Processing region 91
Process 586247 - Completed region 0
Process 586247 - Processing region 1
Process 586248 - Completed region 13
Process 586248 - Processing region 14
Process 586250 - Completed region 39
Process 586250 - Processing region 40
Process 586252 - Completed region 65
Process 586252 - Processing region 66
Process 586251 - Completed region 52
Process 586251 - Processing region 53
Process 586249 - Completed region 26
Process 586249 - Processing region 27
Process 586254 - Completed region 91
Process 586254 - Processing region 92
Process 586253 - Completed region 78
Process 586253 - Processing region 79
Process 586247 - Completed region 1
Process 586247 - Processing region 2
Proce

In [3]:
from statsmodels.stats.multitest import multipletests

# 进行 FDR 校正（对所有脑区的 Time_Condition_p 进行校正）
results["P_Value_Perm_FDR"] = multipletests(results["P_Value_Perm"], method="fdr_bh")[1]
significant_regions1 = results[
    (results["P_Value_Perm_FDR"] < 0.05)]

print(significant_regions1)
# 提取符合条件的脑区编号
significant_region_numbers = significant_regions1["Brain_Region"].tolist()

# 输出脑区编号
print(significant_region_numbers)

     Brain_Region                  Mean_Y             P_Value_Perm  \
2               2   [0.02485429898598831]  [0.0013333333333333333]   
78             78   [0.03097278704434006]                    [0.0]   
79             79    [0.0381292911557238]                    [0.0]   
82             82   [0.02131234500600297]                  [0.002]   
88             88  [0.018136254548859598]                    [0.0]   
97             97  [0.018473511523793285]  [0.0006666666666666666]   
100           100  [0.026113855610189193]  [0.0013333333333333333]   
139           139  [0.030628169897174453]  [0.0003333333333333333]   
140           140  [0.049534986654265784]                    [0.0]   
141           141   [0.05234515527319952]                    [0.0]   
142           142   [0.04288655666864402]                    [0.0]   
143           143  [0.028230221597956018]                    [0.0]   
347           347  [0.029362782235946072]  [0.0016666666666666668]   
349           349  [

/home/sylsherry/miniconda3/envs/tfgpu/lib/python3.9/site-packages/statsmodels/stats/multitest.py:355: RuntimeWarning: invalid value encountered in divide
  pvals_corrected_raw = pvals_sorted / ecdffactor


In [16]:
# 保存为 CSV 文件
results.to_csv("sicheck_0526.csv", index=False)

print("Results saved to 'sicheck_0526.csv'")

Results saved to 'sicheck_0526.csv'
